# Two Paths to Tools

In this notebook you will explore the **dual-path architecture** that your enterprise Agentic AI platform provides for tool access.

Module 3 deployed a fully functional MCP Gateway and Registry backed by NGINX and CloudFront. That infrastructure gives agents fast, direct access to MCP-compliant tool servers. Module 4 adds a second, **governed** path through AWS Bedrock AgentCore Gateway that layers on audit logging, guardrails, and support for tool types that NGINX cannot serve (Lambda functions, external APIs).

By the end of this notebook you will understand:
- What each path provides
- How to query the Registry API that feeds both paths
- When to choose direct access vs. governed access

## Architecture Overview

```
                         +---------------------------+
                         |   MCP Registry (Module 3) |
                         |   Tool catalog, metadata  |
                         +---------------------------+
                                    |
                    +---------------+---------------+
                    |                               |
             PATH A (Direct)                 PATH B (Governed)
                    |                               |
         +-------------------+          +------------------------+
         |  CloudFront/ALB   |          |  AgentCore Gateway     |
         |  NGINX Reverse    |          |  - JWT Auth (Cognito)  |
         |  Proxy            |          |  - Request Interceptor |
         |                   |          |  - Response Interceptor|
         +-------------------+          |  - Bedrock Guardrails  |
                    |                   +------------------------+
                    |                        |            |
         +-------------------+     +-----------+  +-------------+
         | Docker MCP        |     | Lambda    |  | HTTP/API    |
         | Servers           |     | Targets   |  | Targets     |
         | (SSE transport)   |     | (native)  |  | (external)  |
         +-------------------+     +-----------+  +-------------+
```

**Path A -- Direct via NGINX (Module 3)**
- Agent connects through CloudFront to NGINX, which reverse-proxies to Docker-based MCP servers.
- Low latency, no extra processing. Uses MCP SSE transport.
- Limited to tools that run as Docker containers behind NGINX.

**Path B -- Governed via AgentCore Gateway (Module 4)**
- Agent connects through the AgentCore Gateway, which validates JWT tokens from Cognito, runs request/response interceptors, and dispatches to targets.
- Supports Lambda-native targets (`lambda://` URLs) and external HTTP APIs -- tool types that NGINX cannot route.
- Every request is audit-logged by the request interceptor. Every response can be screened by Bedrock Guardrails via the response interceptor.
- Higher latency, but adds governance, compliance, and extensibility.

## Step 1: Discover What Module 3 Provides

Module 3's CloudFormation stacks export several values that both paths rely on. Let's fetch them to see the infrastructure that is already in place.

In [ ]:
import boto3, json

region = boto3.session.Session().region_name or "us-west-2"
cfn = boto3.client("cloudformation", region_name=region)

# Fetch all exports and filter for the ones Module 3 creates
exports = {}
paginator = cfn.get_paginator("list_exports")
for page in paginator.paginate():
    for export in page["Exports"]:
        exports[export["Name"]] = export["Value"]

# The key exports from Module 3 (environment name = "workshop")
keys_of_interest = [
    "workshop-RegistryUrl",
    "workshop-MainCloudFrontUrl",
    "workshop-CognitoUserPoolId",
    "workshop-CognitoM2MClientId",
    "workshop-CognitoDomain",
]

print("Module 3 CloudFormation Exports")
print("=" * 60)
for key in keys_of_interest:
    value = exports.get(key, "(not found)")
    print(f"  {key:40s} = {value}")

# Store for later cells
REGISTRY_URL = exports.get("workshop-RegistryUrl", "")
CLOUDFRONT_URL = exports.get("workshop-MainCloudFrontUrl", "")
COGNITO_POOL_ID = exports.get("workshop-CognitoUserPoolId", "")
M2M_CLIENT_ID = exports.get("workshop-CognitoM2MClientId", "")

# Preflight — fail fast if any required export is missing
missing = [k for k, v in {
    "workshop-RegistryUrl": REGISTRY_URL,
    "workshop-MainCloudFrontUrl": CLOUDFRONT_URL,
    "workshop-CognitoUserPoolId": COGNITO_POOL_ID,
    "workshop-CognitoM2MClientId": M2M_CLIENT_ID,
}.items() if not v]
if missing:
    raise RuntimeError(
        f"Required Module 3 exports missing: {missing}. "
        f"Verify workshop-registry-stack and workshop-agentcore-stack are CREATE_COMPLETE."
    )

## Step 2: Test Path A -- Direct Access via the Registry API

Path A is already live. The Registry API sits behind CloudFront and NGINX. Let's authenticate with the Registry API token (stored in a secret referenced by the `workshop-RegistryApiTokenSecretArn` CloudFormation export) and list the registered MCP servers.

This call goes: **your notebook -> CloudFront -> ALB -> NGINX -> Registry API (FastAPI)**.

In [ ]:
import httpx

# Get the Registry API token from the dedicated secret
sm = boto3.client("secretsmanager", region_name=region)
api_token_arn = exports.get("workshop-RegistryApiTokenSecretArn", "")
if not api_token_arn:
    raise RuntimeError(
        "Export 'workshop-RegistryApiTokenSecretArn' not found. "
        "Is workshop-registry-stack deployed and CREATE_COMPLETE?"
    )
api_secret = json.loads(
    sm.get_secret_value(SecretId=api_token_arn)["SecretString"]
)
access_token = api_secret.get("api_token", "")
headers = {"Authorization": f"Bearer {access_token}"}
print(f"Using Registry API token for access")

# List all registered MCP servers via the Registry API
servers_resp = httpx.get(f"{REGISTRY_URL}/api/servers", headers=headers, timeout=15)
servers_resp.raise_for_status()
servers = servers_resp.json()

if isinstance(servers, dict):
    servers = servers.get("servers", servers.get("items", []))

print(f"\nRegistry contains {len(servers)} server(s):")
print("-" * 60)
for s in servers:
    name = s.get("display_name") or s.get("server_name") or s.get("name", "?")
    proxy = s.get("proxy_pass_url", "?")
    tags = s.get("tags", "")
    print(f"  {name:30s}  proxy={proxy}")
    if tags:
        print(f"  {'':30s}  tags={tags}")

## Decision Framework: When to Use Each Path

| Criteria | Path A (NGINX / Direct) | Path B (AgentCore Gateway) |
|---|---|---|
| **Tool type** | Docker-based MCP servers only | Lambda functions, HTTP APIs, and Docker servers |
| **Transport** | MCP SSE via NGINX reverse proxy | AgentCore managed (Lambda invoke, HTTP) |
| **Authentication** | Cognito JWT via NGINX auth_request | Cognito JWT via Gateway custom authorizer |
| **Audit logging** | NGINX access logs | Lambda request interceptor (structured, per-request) |
| **Content guardrails** | None | Bedrock Guardrails via response interceptor |
| **Latency** | Lower (single NGINX hop) | Higher (interceptor Lambdas + gateway routing) |
| **Best for** | High-throughput, trusted internal tools | Regulated workloads, external APIs, Lambda tools |

**In practice, both paths coexist.** The Sync Lambda reads from the same Registry and creates AgentCore Gateway targets for every registered server -- including the Docker-based ones that Path A already serves. This means agents can choose the path that fits their requirements at invocation time.

### What's Next

In the following notebooks you will:
1. **Deploy the CDK stack** that creates the Lambda functions, IAM role, and EventBridge rule for Path B.
2. **Create the AgentCore Gateway** with Cognito auth and interceptors.
3. **Register new tool types** (Lambda, OpenAPI) that only Path B can serve.
4. **Sync the Registry catalog** into AgentCore Gateway targets.